# Business Intelligence & Data Engineering
---
**MSc in Business Analytics** | *Athens University of Economics and Business*  
**Assignment:** Apache Spark  
**Student:** Anastasios Rigos (f2822511)

Apache Spark run on Java 17 or Java 21, we will use Java 21.

In [1]:
import os

JAVA_HOME = "/opt/homebrew/Cellar/openjdk@21/21.0.10/libexec/openjdk.jdk/Contents/Home"
os.environ["JAVA_HOME"] = JAVA_HOME

In [2]:
! echo $JAVA_HOME

/opt/homebrew/Cellar/openjdk@21/21.0.10/libexec/openjdk.jdk/Contents/Home


In [3]:
! java -version

openjdk version "21.0.10" 2026-01-20
OpenJDK Runtime Environment Homebrew (build 21.0.10)
OpenJDK 64-Bit Server VM Homebrew (build 21.0.10, mixed mode, sharing)


# Part 1 - SparkSQL

We set up the scene by importing from the class SparkSession from the pyspark Library  

In [4]:
from pyspark.sql import SparkSession

We create a Spark Session for the Q-Commerce project with Application Name "Q-Commerce-project".

In [5]:
spark = SparkSession.builder.appName("Q-Commerce-project").getOrCreate()

Using Spark's default log4j profile: org/apache/spark/log4j2-defaults.properties
Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
26/04/18 03:37:21 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable


We read the retail-dataset.csv file in the folder ./data and we save it in a dataframe with the name retail_df.

In [6]:
# header=True: the first row of the CSV file contains the column names
# inferSchema=True: automatically infer the data types of the columns
retail_df = spark.read.csv("data/retail-dataset.csv", header=True, inferSchema=True)

We cache the dataframe in memory because we are going to run many queries on it
for questions 1 to 9. Without caching, Spark would read the CSV file from disk
every time we call an action, which would be a bit slower.

In [7]:
retail_df.cache()

DataFrame[Transaction_ID: int, Date: timestamp, Customer_Name: string, Product: string, Total_Items: int, Total_Cost: double, Payment_Method: string, City: string, Store_Type: string, Discount_Applied: boolean, Customer_Category: string, Season: string, Promotion: string]

Note: Spark will cache the dataset in the first action (in question 1) because of the lazy evaluation.

## Question 1: Display descriptive statistics for each column of the dataset.  

We display the descriptive statistics for each column of the dataset using the describe() method in pandas, which provides us the count, the mean, the standard deviation, the minimum and the maximum value of each column.

In [8]:
retail_df.describe().toPandas()

26/04/18 03:37:25 WARN SparkStringUtils: Truncated the string representation of a plan since it was too large. This behavior can be adjusted by setting 'spark.sql.debug.maxToStringFields'.


,summary,Transaction_ID,Customer_Name,Product,Total_Items,Total_Cost,Payment_Method,City,Store_Type,Customer_Category,Season,Promotion
0,count,1000000,1000000,1000000,1000000,1000000,1000000,1000000,1000000,1000000,1000000,1000000
1,mean,1.0004999995E9,NaN,NaN,5.495941,52.455220400000094,NaN,NaN,NaN,NaN,NaN,NaN
2,stddev,288675.2789323441,NaN,NaN,2.871654187209314,27.41698914533317,NaN,NaN,NaN,NaN,NaN,NaN
3,min,1000000000,Aaron Acevedo,"['Air Freshener', 'Air Freshener', 'Air Freshe...",1,5.0,Cash,Atlanta,Convenience Store,Homemaker,Fall,BOGO (Buy One Get One)
4,max,1000999999,Zoe York,['Yogurt'],10,100.0,Mobile Payment,Seattle,Warehouse Club,Young Adult,Winter,None


Alternatively, we can use the summary() method, which provide also the 25%, 50%, 75% quartiles for the numeric variables

In [9]:
retail_df.summary().toPandas()

,summary,Transaction_ID,Customer_Name,Product,Total_Items,Total_Cost,Payment_Method,City,Store_Type,Customer_Category,Season,Promotion
0,count,1000000,1000000,1000000,1000000,1000000,1000000,1000000,1000000,1000000,1000000,1000000
1,mean,1.0004999995E9,NaN,NaN,5.495941,52.455220400000094,NaN,NaN,NaN,NaN,NaN,NaN
2,stddev,288675.2789323441,NaN,NaN,2.871654187209314,27.41698914533317,NaN,NaN,NaN,NaN,NaN,NaN
3,min,1000000000,Aaron Acevedo,"['Air Freshener', 'Air Freshener', 'Air Freshe...",1,5.0,Cash,Atlanta,Convenience Store,Homemaker,Fall,BOGO (Buy One Get One)
4,25%,1000249917,NaN,NaN,3,28.71,NaN,NaN,NaN,NaN,NaN,NaN
5,50%,1000500057,NaN,NaN,5,52.42,NaN,NaN,NaN,NaN,NaN,NaN
6,75%,1000749990,NaN,NaN,8,76.18,NaN,NaN,NaN,NaN,NaN,NaN
7,max,1000999999,Zoe York,['Yogurt'],10,100.0,Mobile Payment,Seattle,Warehouse Club,Young Adult,Winter,None


As we can see the dataset contains 1,000,000 rows of transcations. The numeric columns consist of Transaction_ID, Total_Items and Total_Cost and the categorical ones are Customer_Name, Product

## Question 2. Identify the customers that pay in cash and display the two corresponding columns.  

Let's see which are the types of the Payment Method.

In [10]:
# Select distinct values from the Payment_Method column and convert the result to a Pandas DataFrame
retail_df.select("Payment_Method").distinct().toPandas()

,Payment_Method
0,Mobile Payment
1,Credit Card
2,Cash
3,Debit Card


We see that that the Payment Method for paying in cash is "Cash".  
So we want to find the transactions that the customer paid with cash and display only the two corresponding columns.

In [11]:
# Filter the DataFrame to include only rows where the Payment_Method is 'Cash',
# select only the Customer_Name and Payment_Method columns 
# and convert the resulting DataFrame to a Pandas DataFrame
retail_df.filter("Payment_Method == 'Cash'") \
    .select("Customer_Name", "Payment_Method") \
    .toPandas()

,Customer_Name,Payment_Method
0,Michelle Carlson,Cash
1,Joshua Frazier,Cash
2,Victoria Garrett,Cash
3,Angela Jones,Cash
4,Don Harris,Cash
...,...,...
250225,Joshua Green,Cash
250226,Leslie Sanford,Cash
250227,Gary Mosley,Cash
250228,James Villegas,Cash


We have 250230 transactions, where customers paid in cash with their corresponding names.

## Question 3: Identify and display the purchases carried out by students or teenagers.  

Let's see which values in Customer_Category column corresponds to students and teenagers

In [12]:
# Find the distinct values in the Customer_Category column.
retail_df.select("Customer_Category").distinct().toPandas()

,Customer_Category
0,Teenager
1,Student
2,Retiree
3,Homemaker
4,Senior Citizen
5,Young Adult
6,Professional
7,Middle-Aged


So, we want to filder the dataset and keep only the purschases having Customer_Category value of "Teemager" or "Student"

In [13]:
# Filter the DataFrame to include only rows where the Customer_Category is 'Teenager' or 'Student'
retail_df.filter((retail_df.Customer_Category == 'Teenager') | (retail_df.Customer_Category == 'Student')) \
    .toPandas()

,Transaction_ID,Date,Customer_Name,Product,Total_Items,Total_Cost,Payment_Method,City,Store_Type,Discount_Applied,Customer_Category,Season,Promotion
0,1000000006,2023-01-08 10:40:03,Victoria Garrett,"['Honey', 'BBQ Sauce', 'Soda', 'Olive Oil', 'G...",4,5.28,Cash,Boston,Specialty Store,False,Student,Summer,Discount on Selected Items
1,1000000012,2023-05-27 15:52:59,Sheila Mcgee,"['Salmon', 'Shaving Cream']",9,39.75,Debit Card,New York,Pharmacy,False,Student,Summer,Discount on Selected Items
2,1000000020,2021-10-31 07:46:22,Angela Jones,"['Pancake Mix', 'Vacuum Cleaner']",7,83.86,Cash,Houston,Pharmacy,False,Teenager,Winter,Discount on Selected Items
3,1000000024,2023-02-27 16:46:56,Hannah Rowland,"['Sponges', 'Vacuum Cleaner', 'Iron']",4,24.03,Debit Card,Dallas,Pharmacy,False,Student,Winter,None
4,1000000028,2022-08-13 01:26:19,Clayton Roth,"['Tomatoes', 'Salmon']",7,21.08,Credit Card,San Francisco,Specialty Store,True,Teenager,Fall,None
...,...,...,...,...,...,...,...,...,...,...,...,...,...
250156,1000999972,2020-05-24 19:16:30,William Brown,['Shaving Cream'],8,32.64,Credit Card,Houston,Specialty Store,False,Teenager,Fall,Discount on Selected Items
250157,1000999977,2023-10-05 09:53:09,Robin Waters,"['Tuna', 'Orange', 'Ketchup']",8,87.59,Mobile Payment,Los Angeles,Supermarket,True,Student,Summer,BOGO (Buy One Get One)
250158,1000999983,2023-10-02 22:34:12,Mr. James Buchanan,"['Paper Towels', 'Tomatoes']",2,74.41,Credit Card,Atlanta,Warehouse Club,False,Student,Spring,BOGO (Buy One Get One)
250159,1000999988,2023-06-08 01:04:43,Blake Richard,"['Chips', 'Razors', 'Tea', 'Tomatoes']",9,94.09,Mobile Payment,Atlanta,Convenience Store,False,Student,Summer,BOGO (Buy One Get One)


## Question 4: Identify and display the customer categories included in the dataset and the number of purchases per customer category.  

As we did for the cell above, we find the customer categories by finding the unique/distict values of the category Customer_Category column.

In [14]:
# Select the Customer_Category column and find the distinct values of the column.
retail_df.select("Customer_Category").distinct().toPandas()

,Customer_Category
0,Teenager
1,Student
2,Retiree
3,Homemaker
4,Senior Citizen
5,Young Adult
6,Professional
7,Middle-Aged


We have eight different customer categories, as we can see from the Dataframe. 

- Teenager
- Student 
- Retiree
- Homemaker 
- Senior Citizen
- Young Adult 
- Professional
- Middle-Aged

Let's now find the the number of purchases per customer category. We need to group by the category of the customer and calculate the transactions/purchases per category.

In [15]:
# Group the DataFrame by the Customer_Category column,
# count the number of occurrences of each category,
# sort the results in descending order based on the count 
# and convert the result to a Pandas DataFrame
retail_df.groupBy("Customer_Category") \
    .count() \
    .sort("count", ascending=False) \
    .toPandas()

,Customer_Category,count
0,Senior Citizen,125485
1,Homemaker,125418
2,Teenager,125319
3,Retiree,125072
4,Student,124842
5,Professional,124651
6,Middle-Aged,124636
7,Young Adult,124577


We can see from the Datadrame the number of purchses for every customer category, with Senior Citizens having the most purchses in the datasets with 125485 transactions, while the sutomer category with the least purchases (124,577) are the Young Adults.  

**Note:** Sorting the categories by their count is not necessary, but it provides a clearer view of the results, since we can immediately see which customer categories are the most and least frequent in the dataset.

## Question 5: Identify the purchases that included a promotion and display the 10 most recent ones.  

We have find which are the different types of the promotions.

In [16]:
# Find the different promotions in the Promotion column
retail_df.select("Promotion").distinct().toPandas()

,Promotion
0,BOGO (Buy One Get One)
1,None
2,Discount on Selected Items


The Promotion column containd 3 unique values: 
- BOGO (Buy One Get One)
- None
- Discount on Selected Items  

In the dataset we have also another column with the name "Discount Applied", so let's see what this column contains

In [17]:
retail_df.select("Discount_Applied").distinct().toPandas()

,Discount_Applied
0,True
1,False


This column consists of True or False, whether in the purchase there was a discount or not.

Let's now see if True in the column Discount means that we have a Promotion also.

In [18]:
retail_df.groupBy("Discount_Applied", "Promotion").count().toPandas()

,Discount_Applied,Promotion,count
0,True,None,166759
1,False,BOGO (Buy One Get One),166208
2,False,None,167184
3,False,Discount on Selected Items,166504
4,True,BOGO (Buy One Get One),166479
5,True,Discount on Selected Items,166866


We see that Discount_Applied is different from Promotion, the fact that in a purchase there is a discount does not mean that we have a promotion also. Therefore, for the purpose of the question we will use the column Promotion.

So, we have to filter the dataset to include only the rows with Promotion BOGO (Buy One Get One) or Discount on Selected Items or another idea is to exlude the rows when Promotion equals None.

We will filter the dataset to include only the rows with Promotion not None.

In [19]:
retail_df.filter(retail_df.Promotion != "None") \
    .orderBy(retail_df.Date.desc()) \
    .limit(10).toPandas()

,Transaction_ID,Date,Customer_Name,Product,Total_Items,Total_Cost,Payment_Method,City,Store_Type,Discount_Applied,Customer_Category,Season,Promotion
0,1000008443,2024-05-18 19:29:37,Abigail Villarreal,"['Tuna', 'Toothbrush', 'Orange']",5,6.22,Mobile Payment,Atlanta,Department Store,True,Retiree,Spring,Discount on Selected Items
1,1000442620,2024-05-18 19:28:08,Devon Hooper,['Soap'],10,86.27,Mobile Payment,Boston,Pharmacy,True,Teenager,Summer,BOGO (Buy One Get One)
2,1000862966,2024-05-18 19:26:29,Emily Melendez,"['Dustpan', 'Coffee', 'Light Bulbs', 'Razors']",5,70.85,Credit Card,Los Angeles,Pharmacy,False,Middle-Aged,Summer,Discount on Selected Items
3,1000298374,2024-05-18 19:23:19,Bryan Mclaughlin,"['Vacuum Cleaner', 'Garden Hose']",8,93.32,Mobile Payment,Boston,Convenience Store,False,Homemaker,Spring,Discount on Selected Items
4,1000060432,2024-05-18 19:19:33,Tiffany Peters,"['Tomatoes', 'Power Strips', 'Rice']",6,46.28,Debit Card,Boston,Convenience Store,False,Retiree,Summer,Discount on Selected Items
5,1000963132,2024-05-18 19:18:20,Andrea Glover,"['Paper Towels', 'Toothbrush', 'Cleaning Rags']",9,36.69,Cash,Dallas,Pharmacy,True,Middle-Aged,Spring,Discount on Selected Items
6,1000581562,2024-05-18 19:16:00,Tiffany Taylor,"['Apple', 'Shrimp', 'BBQ Sauce']",3,12.84,Cash,Boston,Department Store,True,Teenager,Fall,Discount on Selected Items
7,1000247237,2024-05-18 19:06:29,Gary Stanley,"['Eggs', 'Lawn Mower']",2,34.97,Cash,Seattle,Department Store,True,Professional,Spring,Discount on Selected Items
8,1000792049,2024-05-18 19:05:06,Travis Carlson,"['Cereal', 'Toilet Paper', 'Syrup', 'Trash Can...",6,78.46,Cash,San Francisco,Specialty Store,True,Student,Fall,BOGO (Buy One Get One)
9,1000424076,2024-05-18 18:53:32,Lisa Johnson,['Diapers'],10,31.08,Credit Card,San Francisco,Pharmacy,True,Student,Summer,BOGO (Buy One Get One)


## Question 6: Find the distinct cities where purchases have occured and order the cities by the number of purchases that have happened in each city from highest to lowest.  

We have to group the dataset by the different cities, count the number of purchases for every city and then sort the cities in descending order of the number of purchases.

In [20]:
# Group the DataFrame by the City column,
# count the number of occurrences of each city and remane the column to Number of Purchases
# sort the results in descending order based on the Number of Purchases
# and convert the result to a Pandas DataFrame
retail_df.groupBy("City") \
    .count().withColumnRenamed("count","Number of Purchases") \
    .sort("count", ascending=False) \
    .toPandas()

,City,Number of Purchases
0,Boston,100566
1,Dallas,100559
2,Seattle,100167
3,Chicago,100059
4,Houston,100050
5,New York,100007
6,Los Angeles,99879
7,Miami,99839
8,San Francisco,99808
9,Atlanta,99066


As we can see from the output, purchases occured in 10 different/distict cities with Boston leading with 100,566 purchases.  

**Note:** The rename of the column count to Number of Purchases is not necessary but this way the results provide more information. 

## Question 7: Find the top ten days with the highest sales in the whole time period.  

We need to calculate the sum of the column Total_Sales for every different day in the dataset, however the Date column in our dataset is not in the format YYYT-MM-DD but it is a timestamp in the format YYYY-MM-DD HH:MM:SS. 

So, we have to parse the timestamp to produce a Date representing a specific date in the format YYYY-MM-DD. We will do this by sing the function .todate

In [21]:
import pyspark.sql.functions as fs

retail_df.groupBy(fs.to_date(retail_df.Date).alias("Date")) \
    .agg(fs.sum("Total_Cost").alias("Total_Sales")) \
    .orderBy("Total_Sales", ascending=False) \
    .limit(10) \
    .toPandas()

,Date,Total_Sales
0,2022-06-07,37771.85
1,2020-06-03,37254.95
2,2024-01-21,37109.42
3,2024-04-23,37039.89
4,2021-03-25,36772.45
5,2022-03-10,36764.61
6,2022-05-31,36629.91
7,2021-03-21,36601.14
8,2021-10-03,36598.27
9,2021-03-03,36575.29


## Question 8: Find how much money on average customers in Los Angeles tend to spend in different store types.  

We have to filter our dataset in order to include only the customers in Los Angeles, and then based on these filtered rows we have to group by the different store types and calculate the mean for every group.  

In [22]:
# We filter the dataset to keep only purchases made in Los Angeles, 
# then we group by Store_Type, 
# compute the average of Total_Cost for each group
# and rename to Average Total Cost
retail_df.filter(retail_df.City =='Los Angeles') \
    .groupBy("Store_Type") \
    .mean("Total_Cost").withColumnRenamed("avg(Total_Cost)", "Average Total Cost") \
    .toPandas()

,Store_Type,Average Total Cost
0,Warehouse Club,52.272274
1,Supermarket,52.566340
2,Department Store,52.356963
3,Convenience Store,52.262503
4,Pharmacy,52.715907
5,Specialty Store,52.150182


We see that the customers in Los Angeles on average spend the most in Pharmacy with 52.71, while the least in Specialty Store with 52.15

**Note:** We remaned the column from avg(Total_Cost) to Average Total Cost for better readability.

## Question 9: Identify the top professional customer per city based on the total amount spent.  



We see again what are the customer categories again. 

In [23]:
retail_df.select("Customer_Category").distinct().toPandas()

,Customer_Category
0,Teenager
1,Student
2,Retiree
3,Homemaker
4,Senior Citizen
5,Young Adult
6,Professional
7,Middle-Aged


In [24]:
# Filter the DataFrame to include only rows where the Customer_Category is 'Professional',
# group the resulting DataFrame by the City and Customer_Name columns
# calculate the sum of the Total_Cost column for each group,
# (i.e., for each combination of City and Customer_Name), 
# sort the results first by City in ascending order and then by the sum of Total_Cost in descending order,
# drop duplicate rows based on the City column (i.e., keep only one row for each city)
# and convert the final result to a Pandas DataFrame
retail_df.filter("Customer_Category == 'Professional'") \
    .groupBy("City", "Customer_Name") \
    .sum("Total_Cost") \
    .orderBy("City", "sum(Total_Cost)", ascending=[True, False]) \
    .drop_duplicates(["City"]) \
    .toPandas()

,City,Customer_Name,sum(Total_Cost)
0,Atlanta,Nicholas Smith,336.50
1,Boston,Kevin Smith,551.64
2,Chicago,Michael Smith,520.93
3,Dallas,Sarah Johnson,407.96
4,Houston,David Johnson,380.64
5,Los Angeles,Michael Martin,505.82
6,Miami,Robert Smith,451.32
7,New York,Michael Smith,418.82
8,San Francisco,Michael Jones,367.57
9,Seattle,Michael Smith,445.31


In [25]:
from pyspark.sql.functions import col, sum as _sum, row_number
from pyspark.sql.window import Window

# Step 1, filter only professional customers and compute total spent per customer per city
totals = retail_df.filter(col("Customer_Category") == "Professional") \
                  .groupBy("City", "Customer_Name") \
                  .agg(_sum("Total_Cost").alias("Total_Spent"))

# Step 2, define a window that partitions by city and orders customers by total spent descending
window_spec = Window.partitionBy("City").orderBy(col("Total_Spent").desc())

# Step 3, give each row a rank inside its city and keep only the top one
top_per_city = totals.withColumn("rank", row_number().over(window_spec)) \
                     .filter(col("rank") == 1) \
                     .drop("rank") \
                     .orderBy("City")

top_per_city.toPandas()

,City,Customer_Name,Total_Spent
0,Atlanta,Nicholas Smith,336.50
1,Boston,Kevin Smith,551.64
2,Chicago,Michael Smith,520.93
3,Dallas,Sarah Johnson,407.96
4,Houston,David Johnson,380.64
5,Los Angeles,Michael Martin,505.82
6,Miami,Robert Smith,451.32
7,New York,Michael Smith,418.82
8,San Francisco,Michael Jones,367.57
9,Seattle,Michael Smith,445.31


In [26]:
retail_df.unpersist()

DataFrame[Transaction_ID: int, Date: timestamp, Customer_Name: string, Product: string, Total_Items: int, Total_Cost: double, Payment_Method: string, City: string, Store_Type: string, Discount_Applied: boolean, Customer_Category: string, Season: string, Promotion: string]

# Part 2 - KMeans Clustering with MLlib

In this part we will try to cluster customer behavior using the KMeans algorithm 
from PySpark MLlib. We will use the following four features as requested by the 
assignment.

1. `Customer_Category` (categorical)
2. `Total_Cost` (numerical, the amount of money spent)
3. `Store_Type` (categorical)
4. `City` (categorical)

We treat each transaction as one observation of customer behavior. The three 
categorical features need to be converted into numerical vectors before we can 
pass them to the KMeans algorithm. For this we will use `StringIndexer` followed 
by `OneHotEncoder`, and then combine everything with `VectorAssembler`. All these 
steps will be put together in a Pipeline.

Finally, we will train the model for K values from 2 up to 5, and evaluate each 
one using the silhouette score with the squared euclidean distance.

In [27]:
from pyspark.ml.feature import StringIndexer, OneHotEncoder

categoricalCols = ["Customer_Category", "Store_Type", "City"]

stringIndexer = StringIndexer(inputCols=categoricalCols, outputCols=[col + "_Index" for col in categoricalCols])
encoder = OneHotEncoder(inputCols=stringIndexer.getOutputCols(), outputCols=[col + "_OHE" for col in categoricalCols])

In [28]:
from pyspark.ml.feature import VectorAssembler

# This includes both the numeric columns and the one-hot encoded binary vector columns in our dataset.
numericCols = ["Total_Cost"]
assemblerInputs = [c + "_OHE" for c in categoricalCols] + numericCols
vecAssembler = VectorAssembler(inputCols=assemblerInputs, outputCol="features")

Build the pipeline

In [29]:
from pyspark.ml import Pipeline

transformation_pipeline = Pipeline()\
  .setStages([stringIndexer, encoder, vecAssembler])
fitted_pipeline = transformation_pipeline.fit(retail_df)

In [30]:
transformed_dataframe = fitted_pipeline.transform(retail_df)
transformed_dataframe.cache()

DataFrame[Transaction_ID: int, Date: timestamp, Customer_Name: string, Product: string, Total_Items: int, Total_Cost: double, Payment_Method: string, City: string, Store_Type: string, Discount_Applied: boolean, Customer_Category: string, Season: string, Promotion: string, Customer_Category_Index: double, Store_Type_Index: double, City_Index: double, Customer_Category_OHE: vector, Store_Type_OHE: vector, City_OHE: vector, features: vector]

In [31]:
from pyspark.ml.clustering import KMeans
from pyspark.ml.evaluation import ClusteringEvaluator

for i in range(2, 6):
    kmeans = KMeans().setK(i).setSeed(1)
    model = kmeans.fit(transformed_dataframe)
    predictions = model.transform(transformed_dataframe)
    evaluator = ClusteringEvaluator()
    silhouette = evaluator.evaluate(predictions)
    print("["+str(i)+":] Silhouette with squared euclidean distance = " + str(silhouette))

26/04/18 03:37:45 WARN MemoryStore: Not enough space to cache rdd_204_4 in memory! (computed 48.6 MiB so far)
26/04/18 03:37:45 WARN BlockManager: Persisting block rdd_204_4 to disk instead.
26/04/18 03:37:45 WARN MemoryStore: Not enough space to cache rdd_204_4 in memory! (computed 48.6 MiB so far)
26/04/18 03:37:45 WARN MemoryStore: Not enough space to cache rdd_204_4 in memory! (computed 48.6 MiB so far)
26/04/18 03:37:45 WARN MemoryStore: Not enough space to cache rdd_204_4 in memory! (computed 48.6 MiB so far)
26/04/18 03:37:45 WARN MemoryStore: Failed to reserve initial memory threshold of 1024.0 KiB for computing block rdd_228_9 in memory.
26/04/18 03:37:45 WARN MemoryStore: Failed to reserve initial memory threshold of 1024.0 KiB for computing block rdd_228_2 in memory.
26/04/18 03:37:45 WARN MemoryStore: Not enough space to cache rdd_228_9 in memory! (computed 760.0 B so far)
26/04/18 03:37:45 WARN MemoryStore: Failed to reserve initial memory threshold of 1024.0 KiB for compu

[27.815s][warning][gc,alloc] Executor task launch worker for task 4.0 in stage 112.0 (TID 493): Retried waiting for GCLocker too often allocating 20825 words
[27.815s][warning][gc,alloc] Executor task launch worker for task 0.0 in stage 112.0 (TID 489): Retried waiting for GCLocker too often allocating 18688 words


26/04/18 03:37:48 WARN MemoryStore: Not enough space to cache rdd_204_9 in memory! (computed 36.9 MiB so far)
26/04/18 03:37:48 WARN MemoryStore: Not enough space to cache rdd_204_6 in memory! (computed 48.6 MiB so far)
26/04/18 03:37:49 WARN MemoryStore: Not enough space to cache rdd_204_9 in memory! (computed 36.9 MiB so far)
26/04/18 03:37:49 WARN MemoryStore: Not enough space to cache rdd_204_6 in memory! (computed 48.6 MiB so far)
26/04/18 03:37:49 WARN MemoryStore: Not enough space to cache rdd_204_6 in memory! (computed 48.6 MiB so far)
26/04/18 03:37:49 WARN MemoryStore: Not enough space to cache rdd_204_9 in memory! (computed 36.9 MiB so far)
26/04/18 03:37:49 WARN MemoryStore: Not enough space to cache rdd_204_9 in memory! (computed 36.9 MiB so far)
26/04/18 03:37:49 WARN MemoryStore: Not enough space to cache rdd_204_6 in memory! (computed 48.6 MiB so far)
26/04/18 03:37:49 WARN MemoryStore: Not enough space to cache rdd_204_6 in memory! (computed 48.6 MiB so far)
26/04/18 0

[2:] Silhouette with squared euclidean distance = 0.7887805601878382


26/04/18 03:37:50 WARN MemoryStore: Not enough space to cache rdd_204_6 in memory! (computed 48.6 MiB so far)
26/04/18 03:37:50 WARN MemoryStore: Failed to reserve initial memory threshold of 1024.0 KiB for computing block rdd_311_8 in memory.
26/04/18 03:37:50 WARN MemoryStore: Failed to reserve initial memory threshold of 1024.0 KiB for computing block rdd_311_2 in memory.
26/04/18 03:37:50 WARN MemoryStore: Not enough space to cache rdd_311_8 in memory! (computed 760.0 B so far)
26/04/18 03:37:50 WARN BlockManager: Persisting block rdd_311_8 to disk instead.
26/04/18 03:37:50 WARN MemoryStore: Not enough space to cache rdd_311_2 in memory! (computed 760.0 B so far)
26/04/18 03:37:50 WARN MemoryStore: Failed to reserve initial memory threshold of 1024.0 KiB for computing block rdd_311_4 in memory.
26/04/18 03:37:50 WARN MemoryStore: Not enough space to cache rdd_311_4 in memory! (computed 760.0 B so far)
26/04/18 03:37:50 WARN BlockManager: Persisting block rdd_311_2 to disk instead.

...

26/04/18 03:37:56 WARN MemoryStore: Not enough space to cache rdd_204_6 in memory! (computed 48.6 MiB so far)
26/04/18 03:37:56 WARN MemoryStore: Failed to reserve initial memory threshold of 1024.0 KiB for computing block rdd_408_4 in memory.
26/04/18 03:37:56 WARN MemoryStore: Failed to reserve initial memory threshold of 1024.0 KiB for computing block rdd_408_9 in memory.
26/04/18 03:37:56 WARN MemoryStore: Not enough space to cache rdd_408_4 in memory! (computed 760.0 B so far)
26/04/18 03:37:56 WARN MemoryStore: Not enough space to cache rdd_408_9 in memory! (computed 760.0 B so far)
26/04/18 03:37:56 WARN MemoryStore: Failed to reserve initial memory threshold of 1024.0 KiB for computing block rdd_408_5 in memory.
26/04/18 03:37:56 WARN MemoryStore: Not enough space to cache rdd_408_5 in memory! (computed 760.0 B so far)
26/04/18 03:37:56 WARN MemoryStore: Failed to reserve initial memory threshold of 1024.0 KiB for computing block rdd_408_7 in memory.
26/04/18 03:37:56 WARN Memo

[4:] Silhouette with squared euclidean distance = 0.7268136608711208


26/04/18 03:38:01 WARN MemoryStore: Not enough space to cache rdd_204_6 in memory! (computed 48.6 MiB so far)
26/04/18 03:38:01 WARN MemoryStore: Failed to reserve initial memory threshold of 1024.0 KiB for computing block rdd_507_3 in memory.
26/04/18 03:38:01 WARN MemoryStore: Not enough space to cache rdd_507_3 in memory! (computed 760.0 B so far)
26/04/18 03:38:01 WARN BlockManager: Persisting block rdd_507_3 to disk instead.
26/04/18 03:38:01 WARN MemoryStore: Failed to reserve initial memory threshold of 1024.0 KiB for computing block rdd_507_7 in memory.
26/04/18 03:38:01 WARN MemoryStore: Not enough space to cache rdd_507_7 in memory! (computed 760.0 B so far)
26/04/18 03:38:01 WARN BlockManager: Persisting block rdd_507_7 to disk instead.
26/04/18 03:38:01 WARN MemoryStore: Failed to reserve initial memory threshold of 1024.0 KiB for computing block rdd_507_8 in memory.
26/04/18 03:38:01 WARN MemoryStore: Not enough space to cache rdd_507_8 in memory! (computed 760.0 B so far)

[5:] Silhouette with squared euclidean distance = 0.710744562778583


26/04/18 03:38:07 WARN MemoryStore: Not enough space to cache rdd_204_1 in memory! (computed 48.6 MiB so far)
